# Space Photonics Digital Twin - Interactive Demo

Run each cell with the ▶️ button or Ctrl+Enter

In [ ]:
# Cell 1: Clone repo and setup
!git clone https://github.com/futudmh-svg/Space-Photonics.git
%cd Space-Photonics/space-photonics/twin
import sys
sys.path.insert(0, '.')

In [ ]:
# Cell 2: Imports
import numpy as np
import matplotlib.pyplot as plt
from twin import (
    DigitalTwin, TwinConfig,
    OPABeamSteerer, OPAConfig,
    AtmosphericChannel, AtmosphericConfig
)

In [ ]:
# Cell 3: Initialize Digital Twin
config = TwinConfig(
    dt=1e-4,
    log_interval=1e-2,
    tx_power=1.0,
    wavelength=1550e-9,
    enable_tracking=True,
    enable_thermal=True,
    enable_nested_control=True
)

twin = DigitalTwin(config)
print('Digital Twin initialized')
print(f"  Subsystems: {'nested control' if twin.control else 'direct'} | "
      f"{'thermal' if twin.thermal else 'no thermal'} | atmospheric")

In [ ]:
# Cell 4: Run Simulation (fast: 0.01s)
print('Running simulation...')
twin.run(duration=0.01)
print('Done!')

In [ ]:
# Cell 5: Extract Results
times = [d['time'] for d in twin.log_data]
snrs = [d['optical']['snr_db'] for d in twin.log_data]
rx_power = [d['optical']['rx_power_dbm'] for d in twin.log_data]
pointing_error = [d['optical']['pointing_error'] for d in twin.log_data]

# Plot
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

axes[0].plot(times, snrs, 'b-', linewidth=0.8)
axes[0].set_ylabel('SNR [dB]')
axes[0].set_title('Optical Link Performance')
axes[0].grid(True, alpha=0.3)

axes[1].plot(times, rx_power, 'g-', linewidth=0.8)
axes[1].set_ylabel('RX Power [dBm]')
axes[1].grid(True, alpha=0.3)

axes[2].plot(times, pointing_error, 'r-', linewidth=0.8)
axes[2].set_ylabel('Pointing Error [deg]')
axes[2].set_xlabel('Time [s]')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Summary
summary = twin.get_summary()
print('Simulation Summary:')
print('-' * 40)
for key, value in summary.items():
    if isinstance(value, dict):
        print(f'  {key}:')
        for k, v in value.items():
            print(f'    {k}: {v:.3f}' if isinstance(v, float) else f'    {k}: {v}')
    elif isinstance(value, float):
        print(f'  {key:25s}: {value:10.3f}')
    else:
        print(f'  {key:25s}: {value}')

In [ ]:
# Cell 7: OPA Far-Field Pattern
opa = OPABeamSteerer(OPAConfig(wavelength=1550e-9, num_elements=64))
opa.set_steering_angle(15.0)
theta_range = np.linspace(-30, 30, 500)
intensity = opa.compute_farfield(np.radians(theta_range))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(theta_range, 10*np.log10(intensity + 1e-10), 'b-', linewidth=1.0)
ax.set_xlabel('Angle [deg]')
ax.set_ylabel('Intensity [dB]')
ax.set_title('OPA Far-Field Pattern (64 elements, steered to 15°)')
ax.set_ylim([-60, 5])
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 8: Atmospheric Seeing
atm = AtmosphericChannel(AtmosphericConfig())
r0_values = []
for el in range(5, 91, 5):
    r0 = atm.compute_r0(el)
    r0_values.append(r0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(5, 91, 5), r0_values, 'ro-')
ax.set_xlabel('Elevation [deg]')
ax.set_ylabel('Fried Parameter r0 [m]')
ax.set_title('Atmospheric Seeing vs Elevation')
ax.grid(True, alpha=0.3)
plt.show()